<a href="https://colab.research.google.com/github/StephenRajva/Data-Science-Classwork/blob/main/FT_II_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta

# -----------------------------
# 1. CREATE SYNTHETIC DATASET
# -----------------------------

data = {
    "Rating": [5, 4, 0, 6, None, 3, 2, 5],
    "Review_Text": [
        "Great course",
        "Very helpful",
        "Bad content",
        "Excellent",
        None,
        "Average course",
        "Nice explanation",
        "Loved it"
    ],
    "Review_Date": [
        "Yesterday",
        "2 days ago",
        "March 5th, 2023",
        "3 days ago",
        "Today",
        "March 1st, 2023",
        "Yesterday",
        "5 days ago"
    ]
}

df_reviews = pd.DataFrame(data)

print("Original Dataset:\n", df_reviews)


# -----------------------------
# 2. DATA CLEANING
# -----------------------------

# Rating Cleaning
df_reviews['Rating'] = df_reviews['Rating'].apply(
    lambda x: x if x in [1, 2, 3, 4, 5] else np.nan
)

# FIXED: No inplace, assign back properly
df_reviews['Rating'] = df_reviews['Rating'].fillna(df_reviews['Rating'].median())


# Text Cleaning
def clean_text(text):
    if pd.isna(text):
        return ""
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text.lower()

df_reviews['Cleaned_Text'] = df_reviews['Review_Text'].apply(clean_text)


# Date Cleaning
def convert_relative_date(date_str):
    today = datetime.today()

    if date_str.lower() == "today":
        return today
    elif date_str.lower() == "yesterday":
        return today - timedelta(days=1)
    elif "days ago" in date_str:
        days = int(date_str.split()[0])
        return today - timedelta(days=days)
    else:
        date_str = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', date_str)
        return datetime.strptime(date_str, "%B %d, %Y")

df_reviews['Standard_Date'] = df_reviews['Review_Date'].apply(convert_relative_date)

print("\nCleaned Dataset:\n", df_reviews)


# -----------------------------
# 3. DATA WRANGLING
# -----------------------------

DF_Students = pd.DataFrame({
    "Student_ID": [1, 2, 3, 4],
    "Name": ["Alice", "Bob", "Charlie", "David"],
    "Course_ID": [101, 102, 101, 103]
})

DF_Courses = pd.DataFrame({
    "Course_ID": [101, 102, 103],
    "Course_Name": ["Python", "Data Science", "AI"],
    "Instructor": ["John", "Mike", "Sara"]
})

DF_Scores = pd.DataFrame({
    "Student_ID": [1, 2, 3, 4],
    "Score": [85, 90, 78, 88]
})


# Merge DataFrames
merged_df = DF_Students.merge(DF_Courses, on="Course_ID", how="inner")
merged_df = merged_df.merge(DF_Scores, on="Student_ID", how="inner")

print("\nMerged Data:\n", merged_df)


# Grouping and Aggregation
avg_scores = merged_df.groupby("Course_Name")["Score"].mean().reset_index()

print("\nAverage Score per Course:\n", avg_scores)


# -----------------------------
# 4. UPDATE SCORES
# -----------------------------

DF_Scores_Update = pd.DataFrame({
    "Student_ID": [2, 3],
    "Score": [95, 82]
})

DF_Scores.set_index("Student_ID", inplace=True)
DF_Scores_Update.set_index("Student_ID", inplace=True)

DF_Scores.update(DF_Scores_Update)

DF_Scores.reset_index(inplace=True)

print("\nUpdated Scores:\n", DF_Scores)

Original Dataset:
    Rating       Review_Text      Review_Date
0     5.0      Great course        Yesterday
1     4.0      Very helpful       2 days ago
2     0.0       Bad content  March 5th, 2023
3     6.0         Excellent       3 days ago
4     NaN              None            Today
5     3.0    Average course  March 1st, 2023
6     2.0  Nice explanation        Yesterday
7     5.0          Loved it       5 days ago

Cleaned Dataset:
    Rating       Review_Text      Review_Date      Cleaned_Text  \
0     5.0      Great course        Yesterday      great course   
1     4.0      Very helpful       2 days ago      very helpful   
2     4.0       Bad content  March 5th, 2023       bad content   
3     4.0         Excellent       3 days ago         excellent   
4     4.0              None            Today                     
5     3.0    Average course  March 1st, 2023    average course   
6     2.0  Nice explanation        Yesterday  nice explanation   
7     5.0          Loved it  